In [39]:
import requests
import pandas as pd 
from datetime import datetime
import hashlib
from decimal import Decimal
import duckdb 

## API exploration

In [36]:
url = "https://api.open-meteo.com/v1/forecast"
# Rome latidute and longitude
latitude=41.90
longitude=12.49
days=7

today = datetime.today()

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m,precipitation,windspeed_10m",
    "daily": ["sunset", "sunrise", "daylight_duration"],
    "timezone": "Europe/Rome",
    "past_days": days,
    "forecast_days": days
}
response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

def define_type(time, today):
    if time <= today:
        return "past"
    else:
        return "forecast"
    
df_hourly = pd.DataFrame(data["hourly"])
df_hourly["latitude"] = latitude
df_hourly["longitude"] = longitude
df_hourly["time"] = pd.to_datetime(df_hourly["time"])
df_hourly["type_measurement"] = df_hourly["time"].apply(lambda x: define_type(x, today))
df_hourly["time_request"] = today
df_hourly_past = df_hourly[df_hourly["type_measurement"] == "past"]
df_hourly_past = df_hourly_past.drop(columns = ["type_measurement"])
df_hourly_forecast = df_hourly[df_hourly["type_measurement"] == "forecast"]
df_hourly_forecast = df_hourly_forecast.drop(columns = ["type_measurement"])


df_daily = pd.DataFrame(data["daily"])
df_daily["latitude"] = latitude
df_daily["longitude"] = longitude
df_daily['time'] = pd.to_datetime(df_daily["time"])
df_daily["type_measurement"] = df_daily["time"].apply(lambda x: define_type(x, today))
df_daily["time_request"] = today
df_daily_past = df_daily[df_daily["type_measurement"] == "past"]
df_daily_past = df_daily_past.drop(columns = ["type_measurement"])
df_daily_forecast = df_daily[df_daily["type_measurement"] == "forecast"]
df_daily_forecast = df_daily_forecast.drop(columns = ["type_measurement"])




## Database Creation

- Document the schema in docs/schema.md
- git commit -m "docs: database schema design" 



In [37]:
def generate_id(row):
    timestamp = row["time"]
    latitude = row["latitude"]
    longitude = row["longitude"]

    # Normalize inputs
    ts_str = timestamp.isoformat() if isinstance(timestamp, datetime) else str(timestamp)
    latitude_str = format(Decimal(str(latitude)), 'f')
    longitude_str = format(Decimal(str(longitude)), 'f')

    # Combine
    raw = f"{ts_str}|{latitude_str}|{longitude_str}"

    # Hash
    return hashlib.sha256(raw.encode()).hexdigest()


In [38]:
df_hourly_past["id"] = df_hourly_past.apply(generate_id, axis = 1)
df_hourly_forecast["id"] = df_hourly_forecast.apply(generate_id, axis = 1)

df_daily_past["id"] = df_daily_past.apply(generate_id, axis = 1)
df_daily_forecast["id"] = df_daily_forecast.apply(generate_id, axis = 1)

TABLE SCHEMA DF HOURLY

CREATE TABLE IF NOT EXISTS weather_hourly_past (
    time TIMESTAMP,
    temperature_2m DOUBLE,
    precipitation DOUBLE,
    windspeed_10m DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    time_request TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT now(),
    id STRING
    );

CREATE TABLE IF NOT EXISTS weather_hourly_forecast (
    time TIMESTAMP,
    temperature_2m DOUBLE,
    precipitation DOUBLE,
    windspeed_10m DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    time_request TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT now(),
    id STRING
    );




In [46]:
df_daily.head()

,time,sunset,sunrise,daylight_duration,latitude,longitude,type_measurement,time_request
0,2026-04-16,2026-04-16T19:51,2026-04-16T06:28,48202.25,41.9,12.49,past,2026-04-23 12:03:44.155060
1,2026-04-17,2026-04-17T19:52,2026-04-17T06:26,48362.70,41.9,12.49,past,2026-04-23 12:03:44.155060
2,2026-04-18,2026-04-18T19:53,2026-04-18T06:24,48522.79,41.9,12.49,past,2026-04-23 12:03:44.155060
3,2026-04-19,2026-04-19T19:54,2026-04-19T06:23,48682.40,41.9,12.49,past,2026-04-23 12:03:44.155060
4,2026-04-20,2026-04-20T19:56,2026-04-20T06:21,48841.41,41.9,12.49,past,2026-04-23 12:03:44.155060


TABLE SCHEMA DF DAILY

CREATE TABLE IF NOT EXISTS weather_daily_past (
    time TIMESTAMP,
    sunset TIMESTAMP,
    sunrise TIMESTAMP,
    daylight_duration DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    time_request TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT now(),
    id STRING
    );

CREATE TABLE IF NOT EXISTS weather_daily_forecast (
    time TIMESTAMP,
    sunset TIMESTAMP,
    sunrise TIMESTAMP,
    daylight_duration DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    time_request TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT now(),
    id STRING
    );

    

In [50]:
db_path = "../data/weather.duckdb"

conn = duckdb.connect(db_path)

query_create_table_hourly_past = """
                                    CREATE TABLE IF NOT EXISTS weather_hourly_past (
                                        time TIMESTAMP,
                                        temperature_2m DOUBLE,
                                        precipitation DOUBLE,
                                        windspeed_10m DOUBLE,
                                        latitude DOUBLE,
                                        longitude DOUBLE,
                                        time_request TIMESTAMP,
                                        ingested_at TIMESTAMP DEFAULT now(),
                                        id STRING
                                        );
                                    """

query_create_table_hourly_forecast = """
                                    CREATE TABLE IF NOT EXISTS weather_hourly_forecast (
                                        time TIMESTAMP,
                                        temperature_2m DOUBLE,
                                        precipitation DOUBLE,
                                        windspeed_10m DOUBLE,
                                        latitude DOUBLE,
                                        longitude DOUBLE,
                                        time_request TIMESTAMP,
                                        ingested_at TIMESTAMP DEFAULT now(),
                                        id STRING
                                        );
                                    """


query_create_table_daily_past = """ 

                            CREATE TABLE IF NOT EXISTS weather_daily_past (
                                time TIMESTAMP,
                                sunset TIMESTAMP,
                                sunrise TIMESTAMP,
                                daylight_duration DOUBLE,
                                latitude DOUBLE,
                                longitude DOUBLE,
                                time_request TIMESTAMP,
                                ingested_at TIMESTAMP DEFAULT now(),
                                id STRING
                                );

"""

query_create_table_daily_forecast = """ 

CREATE TABLE IF NOT EXISTS weather_daily_forecast (
    time TIMESTAMP,
    sunset TIMESTAMP,
    sunrise TIMESTAMP,
    daylight_duration DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    time_request TIMESTAMP,
    ingested_at TIMESTAMP DEFAULT now(),
    id STRING
    );

"""

conn.execute(query=query_create_table_hourly_past)
conn.execute(query=query_create_table_hourly_forecast)
conn.execute(query=query_create_table_daily_past)
conn.execute(query_create_table_daily_forecast)


query_unique_id_hourly_past = """CREATE UNIQUE INDEX IF NOT EXISTS ux_weather_hourly_past_id
ON weather_hourly_past(id);"""

query_unique_id_hourly_forecast = """CREATE UNIQUE INDEX IF NOT EXISTS ux_weather_hourly_forecast_id
ON weather_hourly_forecast(id);"""

query_unique_id_daily_past = """CREATE UNIQUE INDEX IF NOT EXISTS ux_weather_daily_past_id
ON weather_daily_past(id);"""

query_unique_id_daily_forecast = """CREATE UNIQUE INDEX IF NOT EXISTS ux_weather_daily_forecast_id
ON weather_daily_forecast(id);"""

conn.execute(query=query_unique_id_hourly_past)
conn.execute(query=query_unique_id_hourly_forecast)
conn.execute(query=query_unique_id_daily_past)
conn.execute(query_unique_id_daily_forecast)

conn.close()


## Database Loading

In [51]:
def upsert_weather_hourly(con: duckdb.DuckDBPyConnection, df, table_name: str):
    # optional but recommended: keep the “latest” record per id
    df = df.drop_duplicates(subset=["id"], keep="last")

    con.register("src_df", df)

    con.execute(f"""
        INSERT INTO {table_name} (
            time, temperature_2m, precipitation, windspeed_10m,
            latitude, longitude, time_request, id
        )
        SELECT
            time, temperature_2m, precipitation, windspeed_10m,
            latitude, longitude, time_request, id
        FROM src_df
        ON CONFLICT(id) DO UPDATE SET
            time = excluded.time,
            temperature_2m = excluded.temperature_2m,
            precipitation = excluded.precipitation,
            windspeed_10m = excluded.windspeed_10m,
            latitude = excluded.latitude,
            longitude = excluded.longitude,
            time_request = excluded.time_request,
            ingested_at = now();
    """)

In [52]:
conn = duckdb.connect("../data/weather.duckdb")

upsert_weather_hourly(conn, df_hourly_past, "weather_hourly_past")
upsert_weather_hourly(conn, df_hourly_forecast, "weather_hourly_forecast")



In [55]:
conn.execute("SELECT * FROM weather_hourly_past LIMIT 5").df()

,time,temperature_2m,precipitation,windspeed_10m,latitude,longitude,time_request,ingested_at,id
0,2026-04-16 04:00:00,13.2,0.0,3.2,41.9,12.49,2026-04-23 12:03:44.155060,2026-04-23 12:38:11.760876,35c02ce92fcc09af486afbc052f17cad47d272f4138c7f...
1,2026-04-16 07:00:00,12.4,0.0,2.7,41.9,12.49,2026-04-23 12:03:44.155060,2026-04-23 12:38:11.760876,9269c70c24d85e1f1a5cff3e10fb449c9dbb4478f6bb73...
2,2026-04-17 01:00:00,15.0,0.0,0.0,41.9,12.49,2026-04-23 12:03:44.155060,2026-04-23 12:38:11.760876,3b2c4a32e6c661aa521aaf647294fd42cdbae9b11fee39...
3,2026-04-18 05:00:00,14.0,0.0,3.3,41.9,12.49,2026-04-23 12:03:44.155060,2026-04-23 12:38:11.760876,42b36e93bc2951e20c44bed14a5b29aeabb2af02334804...
4,2026-04-20 01:00:00,15.3,0.0,1.3,41.9,12.49,2026-04-23 12:03:44.155060,2026-04-23 12:38:11.760876,f1416f25903c6d72f92f57cd6fd9dc94ebff73fb0034a6...
